In [1]:
import sys, os, glob, time, socket
from loguru import logger
from tabulate import tabulate
import pandas as pd
import numpy as np
import joblib   
import random

In [2]:
# ANTIGENS = [
#     "Diphtheria",
#     "Pertussis",
#     "Polio",
#     "Tetanus",
#     "Rotavirus",
#     "PCV",
#     "Measles",
#     "Mumps",
#     "Rubella",
#     "Hepatitis_B",
#     "Hib",
#     "HPV",
# ]

ANTIGENS = [
    "Diphtheria",
    "Pertussis",
    "Polio",
    "Tetanus",
    "Rotavirus",
    "PCV",
    "Hepatitis_B",
    "Hib",
    "HPV",
]

N_YEARS = 15
YEARS = np.arange(1, N_YEARS + 1)

# Reformat the capacity scenarios

In [3]:
# capacity_scenario_names = [
#     "base_capacity",
#     "pandemic",
# ]
# capacity_scenario_probs = {
#     "base_capacity": 0.8418,
#     "pandemic": 0.0275,
# }

capacity_scenario_names = [
    "base_capacity",
    "IPV_Shortage",
    "pandemic",
    "funding_delay",
    "innacurate_forecast",
    "supply_chain",
    "other",
]
capacity_scenario_probs = {
    "base_capacity": 0.8418,
    "IPV_Shortage": 0.0173,
    "pandemic": 0.0275,
    "funding_delay": 0.0778,
    "innacurate_forecast": 0.0023,
    "supply_chain": 0.0058,
    "other": 0.0275,
}

# capacity_scenario_names = [
#     "base_capacity"
# ]
# capacity_scenario_probs = {
#     "base_capacity": 1
# }

In [4]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"data\production_capacity_scenarios_15YR_applied.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

print(temp)
# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data\production_capacity_scenarios_15YR_applied.csv", index=False)

       Manufacturer             1  ...  capacity_scenario  capacity_scenario_probability
0       AJ_Vaccines  5.579000e+08  ...              other                         0.0275
1          BB_NCIPD  1.180681e+07  ...              other                         0.0275
2    Bharat_Biotech  7.140000e+07  ...              other                         0.0275
3         Bilthoven  6.335000e+07  ...              other                         0.0275
4      Biological_E  5.207056e+07  ...              other                         0.0275
5    China_National  4.354000e+07  ...              other                         0.0275
6               GSK  2.640000e+07  ...              other                         0.0275
7      Haffkine_Bio  8.120000e+06  ...              other                         0.0275
8           LG_Chem  2.238482e+08  ...              other                         0.0275
9       Merck_Sharp  9.800000e+06  ...              other                         0.0275
10           PT_Bio  

In [5]:
capacity_scenarios

,manufacturer,capacity_scenario,capacity_scenario_probability,capacity
0,Serum_Institute,base_capacity,0.8418,"[5201000.0, 7140000.0, 8329999.999999999, 8329..."
1,GSK,base_capacity,0.8418,"[26400000.0, 20460000.0, 20740000.0, 28500000...."
2,Biological_E,base_capacity,0.8418,"[52270000.0, 57939999.99999999, 58989999.99999..."
3,Haffkine_Bio,base_capacity,0.8418,"[8119999.999999999, 11130000.0, 13020000.0, 12..."
4,Panacea_Biotec,base_capacity,0.8418,"[96900000.0, 110200000.0, 111600000.0, 1053000..."
...,...,...,...,...
100,PT_Bio,other,0.0275,"[53364779.6951938, 65600000.0, 74700000.0, 729..."
101,Panacea_Biotec,other,0.0275,"[96900000.0, 110200000.0, 111600000.0, 1053000..."
102,Pfizer,other,0.0275,"[55510000.0, 63349999.99999999, 65799999.99999..."
103,Sanofi,other,0.0275,"[9730000.0, 10640000.0, 10402228.06457212, 110..."


In [6]:
demand_scenarios_df[demand_scenarios_df['antigen'].isin(ANTIGENS)]

NameError: name 'demand_scenarios_df' is not defined

In [9]:
import pandas as pd
import numpy as np

# Load data
demand_scenarios_df = pd.read_csv(
    "data/Medium_Demand_Structured_15YR.csv",
    converters={"demand": pd.eval},
    usecols=["demand", "antigen", "demand_SID", "probability"]
)

demand_scenarios_df2 = demand_scenarios_df[demand_scenarios_df['antigen'].isin(ANTIGENS)]

# capacity_scenarios_df = pd.read_csv(
#     "data/production_capacity_scenarios.csv",
#     converters={"capacity": pd.eval},
# )

capacity_scenarios_df = capacity_scenarios

# Function to generate pairs
def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 15, verbose: bool = True):
    demand_dict = demand.drop_duplicates(subset=["demand_SID"]).set_index("demand_SID")["probability"].to_dict()
    capacity_dict = capacity.drop_duplicates(subset=["capacity_scenario"]).set_index("capacity_scenario")["capacity_scenario_probability"].to_dict()

    demand_keys = list(demand_dict.keys())
    demand_probs = list(demand_dict.values())

    capacity_keys = list(capacity_dict.keys())
    capacity_probs = list(capacity_dict.values())

    if verbose:
        print(f"Unique demand scenarios: {demand_keys}")
        print(f"Unique capacity scenarios: {capacity_keys}")

    pairs = []
    pair_dfs = []
    prob_dict = {}
    pair_idx = 1

    for selected_capacity, selected_prob_capacity in capacity_dict.items():
        available_demand_keys = demand_keys.copy()
        available_demand_probs = demand_probs.copy()

        for _ in range(n_pairs):
            # Normalize the probabilities
            available_demand_probs = [p / sum(available_demand_probs) for p in available_demand_probs]

            selected_demand = np.random.choice(available_demand_keys, p=available_demand_probs)
            # Combine the probabilities
            selected_prob_demand = demand_dict[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_SID"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["probability", "demand_SID"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demand": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)

            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

            # Remove the selected demand from the available demands
            index = available_demand_keys.index(selected_demand)
            available_demand_keys.pop(index)
            available_demand_probs.pop(index)

    pair_df = pd.concat(pair_dfs, ignore_index=True)

    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())

    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)

    return pairs, pair_df

# Generate pairs
scenario_pairs, pair_df = generate_pairs(demand_scenarios_df2, capacity_scenarios_df, n_pairs=1, verbose=True)

# Export to json and csv
pair_df.to_json("data/pair_demand_capacity_15YR_NO_MMR_stoch.json", orient="records", lines=True)
pair_df.to_csv("data/pair_demand_capacity_15YR_NO_MMR_stoch.csv", index=False)

# Display the DataFrame
pair_df


Unique demand scenarios: [1]
Unique capacity scenarios: ['base_capacity', 'IPV_Shortage', 'pandemic', 'funding_delay', 'innacurate_forecast', 'supply_chain', 'other']


,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[570550152.3038142, 619539915.6026529, 6264554...",antigen,1,Demand: 1 - Capacity: base_capacity,0.8418
1,Tetanus,"[577730183.053466, 626350955.3874788, 63327847...",antigen,1,Demand: 1 - Capacity: base_capacity,0.8418
2,Pertussis,"[320662741.753207, 347414023.65496874, 3524645...",antigen,1,Demand: 1 - Capacity: base_capacity,0.8418
3,Hepatitis_B,"[59502552.14408625, 66824756.29894736, 7094962...",antigen,1,Demand: 1 - Capacity: base_capacity,0.8418
4,Hib,"[522182008.8920537, 582182282.261939, 60863176...",antigen,1,Demand: 1 - Capacity: base_capacity,0.8418
...,...,...,...,...,...,...
163,PT_Bio,"[53364779.6951938, 65600000.0, 74700000.0, 729...",manufacturer,7,Demand: 1 - Capacity: other,0.0275
164,Panacea_Biotec,"[96900000.0, 110200000.0, 111600000.0, 1053000...",manufacturer,7,Demand: 1 - Capacity: other,0.0275
165,Pfizer,"[55510000.0, 63349999.99999999, 65799999.99999...",manufacturer,7,Demand: 1 - Capacity: other,0.0275
166,Sanofi,"[9730000.0, 10640000.0, 10402228.06457212, 110...",manufacturer,7,Demand: 1 - Capacity: other,0.0275


# Create scenario pairs

In [ ]:
pair_df.set_index("pair_idx")

In [ ]:
pair_df.loc[1]

In [ ]:
pair_df['unit'].unique()

In [ ]:
pair_df

In [ ]:
pair_df.groupby('unit').count()